# Analyse Exploratoire — Drone Hubs Ghana

## Contexte

Les zones rurales du Ghana souffrent d'un accès limité aux soins d'urgence. L'isolement géographique et l'état des routes rendent certains villages quasi inaccessibles pour les services de santé conventionnels. Ce projet étudie l'implantation de **hubs de drones médicaux** capables de livrer du sang, des vaccins et des antivenins dans un rayon de **80 km** (autonomie drone : 160 km aller-retour).

## Objectif de ce notebook

Établir un **diagnostic de la situation actuelle** à partir de deux datasets construits par notre pipeline ETL :

- `ghana_villages.csv` — 8 905 villages avec coordonnées, population, élévation et centre de santé le plus proche
- `ghana_health_facilities.csv` — 2 463 infrastructures de santé (hôpitaux, cliniques, pharmacies, CHPS) croisées à partir de 3 sources (OSM, HDX Healthsites, HDX HOT)

On cherche à répondre aux questions suivantes :

1. **Quelle est la répartition géographique** des villages et des centres de santé au Ghana ?
2. **Combien de villages sont isolés** (> 30 km, > 50 km du centre le plus proche) ?
3. **Quelle population est concernée** par cet isolement ?
4. **Où se concentrent les zones mal desservies** ?

Les résultats de cette analyse serviront de **baseline** pour évaluer l'apport du modèle K-Means dans l'étape suivante.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import MarkerCluster

villages = pd.read_csv("../data/processed/ghana_villages.csv")
facilities = pd.read_csv("../data/processed/ghana_health_facilities.csv")

print(f"Villages   : {villages.shape[0]} lignes, {villages.shape[1]} colonnes")
print(f"Facilities : {facilities.shape[0]} lignes, {facilities.shape[1]} colonnes")

## 1. Aperçu des données

In [ ]:
villages.info()
print("\n")
villages.describe()

In [ ]:
villages.head(10)

In [ ]:
print("NaN par colonne :")
print(villages.isna().sum())
print(f"\nDoublons coordonnées : {villages.duplicated(subset=['Latitude', 'Longitude']).sum()}")
print(f"Facility_Type 'Yes'  : {(villages['Nearest_Facility_Type'] == 'Yes').sum()}")

## 2. Répartition des villages par catégorie

Le dataset couvre 10 types de localités, du hameau isolé à la grande ville. On regarde la distribution et la population associée pour comprendre la granularité du réseau à couvrir.

In [ ]:
place_counts = villages["Place_Type"].value_counts().reset_index()
place_counts.columns = ["Place_Type", "Nombre"]

fig = px.bar(
    place_counts,
    x="Place_Type",
    y="Nombre",
    color="Place_Type",
    title="Nombre de localités par catégorie",
    labels={"Place_Type": "Catégorie", "Nombre": "Nombre de localités"},
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Population médiane par catégorie
pop_by_type = (
    villages.groupby("Place_Type")["Population"]
    .agg(["median", "mean", "sum", "count"])
    .sort_values("median", ascending=False)
)
pop_by_type.columns = ["Médiane", "Moyenne", "Total", "Nombre"]
pop_by_type["Médiane"] = pop_by_type["Médiane"].astype(int)
pop_by_type["Moyenne"] = pop_by_type["Moyenne"].astype(int)
pop_by_type["Total"] = pop_by_type["Total"].astype(int)
pop_by_type

**Constat** : les `village/town/hamlet` représentent **89 %** des localités (7 928 sur 8 905) avec une population médiane de 821 habitants. Ce sont les principales cibles du réseau de drones — petites communautés rurales dispersées, souvent sans infrastructure de santé à proximité. Les 14 villes (`city`) et 603 suburbs concentrent la majorité de la population mais sont déjà bien desservis.

## 3. Répartition des infrastructures de santé

2 463 facilities identifiées par le pipeline ETL à partir de 3 sources (OSM Overpass, HDX Healthsites, HDX HOT). Chaque facility est typée selon 9 catégories normalisées et dispose d'un score de confiance (0 à 1).

In [ ]:
fac_counts = facilities["Facility_Type"].value_counts().reset_index()
fac_counts.columns = ["Type", "Nombre"]

fig = px.bar(
    fac_counts,
    x="Type",
    y="Nombre",
    color="Type",
    title="Infrastructures de santé par type (2 463 facilities)",
    labels={"Type": "Type de facility", "Nombre": "Nombre"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Score de confiance par type
conf_by_type = (
    facilities.groupby("Facility_Type")["Confidence_Score"]
    .agg(["mean", "median", "min", "max", "count"])
    .sort_values("mean", ascending=False)
)
conf_by_type.columns = ["Moyenne", "Médiane", "Min", "Max", "Nombre"]
conf_by_type = conf_by_type.round(2)
conf_by_type

Les pharmacies (1 169) et hôpitaux (477) dominent, suivis des cliniques (462). Les CHPS (48) et Health_Post (116) restent sous-représentés malgré leur rôle en zone rurale. Côté confiance, les Doctor (0.77) et Pharmacy (0.76) scorent le plus haut — les Hospital (0.59) souffrent d'un faible taux de multi-sourçage.

## 4. Cartographie — Villages et centres de santé

Carte interactive : les **villages** apparaissent en bleu, les **centres de santé** sont colorés par type. Les marqueurs sont regroupés (clusters) pour la lisibilité. Dézoomer pour voir l'ensemble du Ghana, zoomer pour voir le détail.

In [ ]:
m = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="cartodbpositron")

village_cluster = MarkerCluster(name="Villages")
for _, row in villages.sample(n=2000, random_state=42).iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=2,
        color="#3388ff",
        fill=True,
        fill_opacity=0.5,
        popup=f"{row['Village']} — pop. {row['Population']:.0f}",
    ).add_to(village_cluster)
village_cluster.add_to(m)

fac_colors = {
    "Hospital": "red",
    "Clinic": "orange",
    "Pharmacy": "green",
    "CHPS": "purple",
    "Health_Center": "darkred",
    "Health_Post": "cadetblue",
    "Doctor": "pink",
    "Laboratory": "gray",
    "Other": "lightgray",
}

fac_cluster = MarkerCluster(name="Centres de santé")
for _, row in facilities.iterrows():
    color = fac_colors.get(row["Facility_Type"], "gray")
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['Facility_Name']} ({row['Facility_Type']})",
    ).add_to(fac_cluster)
fac_cluster.add_to(m)

folium.LayerControl().add_to(m)
m

## 5. Analyse des distances — Baseline

C'est le cœur de l'EDA. On mesure l'accès actuel de chaque village au centre de santé le plus proche (distance Haversine). Ces chiffres constituent la **baseline** : la situation *avant* l'implantation de hubs de drones.

In [ ]:
dist = villages["Distance_Nearest_Facility_km"]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Distribution des distances", "Zoom > 20 km"))

fig.add_trace(
    go.Histogram(x=dist, nbinsx=60, marker_color="#3388ff", name="Tous"),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=dist[dist > 20], nbinsx=30, marker_color="#e74c3c", name="> 20 km"),
    row=1, col=2,
)

fig.update_xaxes(title_text="Distance (km)", row=1, col=1)
fig.update_xaxes(title_text="Distance (km)", row=1, col=2)
fig.update_yaxes(title_text="Nombre de villages", row=1, col=1)
fig.update_layout(
    title="Distance au centre de santé le plus proche (Haversine)",
    showlegend=False,
    height=400,
)
fig.show()

In [ ]:
n_total = len(villages)
pop_total = villages["Population"].sum()

seuils = [10, 20, 30, 50, 80]
baseline = []

for s in seuils:
    mask = dist > s
    n = mask.sum()
    pop = villages.loc[mask, "Population"].sum()
    baseline.append({
        "Seuil (km)": f"> {s} km",
        "Villages": n,
        "% villages": f"{100 * n / n_total:.1f} %",
        "Population": f"{pop:,.0f}",
        "% population": f"{100 * pop / pop_total:.2f} %",
    })

baseline_df = pd.DataFrame(baseline)
print(f"Total : {n_total} villages | Population : {pop_total:,.0f}\n")
baseline_df

In [ ]:
# Box plot par catégorie de localité
fig = px.box(
    villages,
    x="Place_Type",
    y="Distance_Nearest_Facility_km",
    color="Place_Type",
    title="Distance au centre de santé le plus proche par type de localité",
    labels={
        "Place_Type": "Catégorie",
        "Distance_Nearest_Facility_km": "Distance (km)",
    },
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30, height=450)
fig.show()

## 6. Cartographie des villages isolés

Les villages à plus de **30 km** du centre de santé le plus proche sont représentés en rouge (> 50 km en noir). Le rayon des cercles est proportionnel à la population.

In [ ]:
m2 = folium.Map(location=[7.95, -1.03], zoom_start=7, tiles="cartodbpositron")

isolated = villages[villages["Distance_Nearest_Facility_km"] > 30].copy()
pop_max = isolated["Population"].max()

for _, row in isolated.iterrows():
    d = row["Distance_Nearest_Facility_km"]
    color = "black" if d > 50 else "#e74c3c"
    radius = max(3, 12 * row["Population"] / pop_max)

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=radius,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=(
            f"<b>{row['Village']}</b><br>"
            f"Pop. {row['Population']:.0f}<br>"
            f"Distance : {d:.1f} km<br>"
            f"Facility : {row['Nearest_Facility_Name']}"
        ),
    ).add_to(m2)

    # Ligne vers le centre de santé le plus proche
    folium.PolyLine(
        locations=[
            [row["Latitude"], row["Longitude"]],
            [row["Nearest_Facility_Latitude"], row["Nearest_Facility_Longitude"]],
        ],
        color=color,
        weight=1,
        opacity=0.3,
    ).add_to(m2)

folium.map.Marker(
    [10.5, -2.3],
    icon=folium.DivIcon(html='<div style="font-size:11px;color:#e74c3c;font-weight:bold">● > 30 km</div>'),
).add_to(m2)
folium.map.Marker(
    [10.3, -2.3],
    icon=folium.DivIcon(html='<div style="font-size:11px;color:black;font-weight:bold">● > 50 km</div>'),
).add_to(m2)

print(f"{len(isolated)} villages isolés (> 30 km)")
m2

## 7. Corrélation distance — élévation — population

On cherche à identifier si les villages isolés partagent des caractéristiques communes. L'hypothèse : l'isolement est-il lié à l'altitude, à la taille du village, ou simplement à la position géographique (latitude/longitude) ?

In [ ]:
fig = px.scatter(
    villages,
    x="Distance_Nearest_Facility_km",
    y="Elevation_m",
    size="Population",
    size_max=15,
    color="Place_Type",
    opacity=0.4,
    title="Distance au centre de santé vs Élévation",
    labels={
        "Distance_Nearest_Facility_km": "Distance (km)",
        "Elevation_m": "Élévation (m)",
        "Place_Type": "Catégorie",
    },
    hover_data=["Village"],
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Matrice de corrélation sur les variables numériques
num_cols = ["Latitude", "Longitude", "Elevation_m", "Population", "Distance_Nearest_Facility_km"]
corr = villages[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Matrice de corrélation",
    labels={"color": "Corrélation"},
)
fig.update_layout(height=450, width=550)
fig.show()

Corrélations faibles sur toute la matrice. La latitude montre un léger lien positif avec la distance (les villages du nord sont plus éloignés), cohérent avec la concentration du réseau de santé dans le sud. Ni l'élévation ni la taille de la population ne discriminent les villages isolés — l'isolement est spatial, pas topographique.

## 8. Score de confiance des facilities reliées

Le pipeline ETL attribue un score de confiance (0–1) à chaque infrastructure de santé selon 5 critères : nombre de sources confirmant son existence, présence d'un nom, type identifié, opérateur renseigné, complétude des données. On vérifie ici la distribution de ce score parmi les facilities auxquelles les villages sont rattachés.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Score de confiance (toutes facilities)",
        "Score de confiance (facilities reliées aux villages)",
    ),
)

fig.add_trace(
    go.Histogram(x=facilities["Confidence_Score"], nbinsx=20, marker_color="#2ecc71", name="Toutes"),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=villages["Nearest_Facility_Confidence"], nbinsx=20, marker_color="#e67e22", name="Reliées"),
    row=1, col=2,
)

fig.update_xaxes(title_text="Score", row=1, col=1)
fig.update_xaxes(title_text="Score", row=1, col=2)
fig.update_yaxes(title_text="Nombre", row=1, col=1)
fig.update_layout(showlegend=False, height=400, title="Distribution du score de confiance")
fig.show()

print(f"Facilities — moyenne : {facilities['Confidence_Score'].mean():.2f}, médiane : {facilities['Confidence_Score'].median():.2f}")
print(f"Reliées    — moyenne : {villages['Nearest_Facility_Confidence'].mean():.2f}, médiane : {villages['Nearest_Facility_Confidence'].median():.2f}")

## 9. Top 20 des villages les plus isolés

In [ ]:
top_isolated = villages.nlargest(20, "Distance_Nearest_Facility_km")[
    ["Village", "Latitude", "Longitude", "Population",
     "Distance_Nearest_Facility_km", "Nearest_Facility_Name", "Nearest_Facility_Type"]
].reset_index(drop=True)

top_isolated.columns = [
    "Village", "Lat", "Lon", "Population",
    "Distance (km)", "Centre le plus proche", "Type"
]
top_isolated

## 10. Synthèse — Baseline

### Chiffres clés

| Métrique | Valeur |
|---|---|
| Villages | 8 905 |
| Facilities | 2 463 (9 types, 3 sources) |
| Distance médiane | 7,07 km |
| Distance moyenne | 9,26 km |
| Distance max | 64,71 km (Kasapuoli, Upper West) |
| Villages > 10 km | 3 109 (34,9 %) |
| Villages > 30 km | 313 (3,5 %) — 423 015 hab. |
| Villages > 50 km | 40 (0,4 %) — 49 612 hab. |
| Villages > 80 km | 0 |

### Observations

1. 75 % des villages sont à moins de 12,6 km d'un centre de santé. Le réseau existant couvre les zones urbaines et périurbaines.

2. Le top 20 des villages isolés se répartit en **deux poches** :
   - **Nord-ouest** (Upper West) : Kasapuoli, Kasana, Nwanduano, Tumu Forestry Reserve → reliés au Chiana Health Centre ou au Hain Polyclinic
   - **Zone forestière** (Brong-Ahafo) : Akyeremade, Lassi, Kwame Danso → reliés à des pharmacies à Yeji ou des cliniques à Kumfia

3. L'isolement n'est pas réservé aux petits hameaux. Kwame Danso (7 994 hab.) et Benyarko (4 116 hab.) figurent parmi les 20 plus isolés.

4. Aucun village ne dépasse 80 km du centre le plus proche. Un rayon d'action de 80 km par hub permet une couverture totale — reste à déterminer le nombre de hubs nécessaire.

5. Le score de confiance moyen des facilities reliées (0,67) est inférieur à la moyenne globale (0,71). Les villages isolés sont rattachés à des facilities moins bien documentées.